# Build Optimized NeuroGolf submission.zip

Scores candidate folders under `solution/`, selects the best ONNX per task, and overwrites `outputs/submission.zip`. This notebook assumes the test cases already pass and only optimizes by estimated NeuroGolf score.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from neurogolf_score import (
    discover_candidate_dirs,
    optimized_summary,
    score_candidate_dirs,
    select_best_by_task,
    validate_submission_zip,
    write_submission_zip,
)

DATA_DIR = repo_root / "neurogolf-2026"
SOLUTION_ROOT = repo_root / "solution"
OUTPUT_ZIP = repo_root / "outputs" / "submission.zip"

# None means all directories directly under solution/. To pin candidates, use e.g. ["6406.16", "6405.61"].
CANDIDATE_NAMES = None

DATA_DIR, SOLUTION_ROOT, OUTPUT_ZIP

(PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/neurogolf-2026'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution'),
 PosixPath('/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/submission.zip'))

In [2]:
candidate_dirs = discover_candidate_dirs(SOLUTION_ROOT, CANDIDATE_NAMES)
print("Candidate folders:")
for path in candidate_dirs:
    print(f"- {path}")

Candidate folders:
- /Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6405.61
- /Users/kaiikeda/Programming/Kaggle/Neuro Golf/solution/6406.16


In [3]:
candidate_summaries, all_rows = score_candidate_dirs(candidate_dirs, DATA_DIR)

try:
    import pandas as pd

    display(pd.DataFrame(candidate_summaries).sort_values("candidate").reset_index(drop=True))
except ImportError:
    candidate_summaries

2026-06-16 22:11:36.201 python[82056:17870034] 2026-06-16 22:11:36.201152 [E:onnxruntime:, inference_session.cc:3540 EndProfiling] Could not write a profile because no model was loaded.
2026-06-16 22:11:58.916 python[82056:17870034] 2026-06-16 22:11:58.915814 [E:onnxruntime:, inference_session.cc:3540 EndProfiling] Could not write a profile because no model was loaded.


,folder,candidate,num_files,num_ok,num_errors,total_score
0,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,6405.61,400,400,0,6405.611247
1,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...,6406.16,400,400,0,6406.160373


In [4]:
selected_rows = select_best_by_task(all_rows)
summary = optimized_summary(selected_rows, all_rows)

print(f"selected tasks : {summary['selected_tasks']}")
print(f"total score    : {summary['total_score']:.6f}")
print(f"selected by candidate: {summary['selected_by_candidate']}")
print(f"selected by status   : {summary['selected_by_status']}")
print(f"error rows           : {len(summary['error_rows'])}")

selected tasks : 400
total score    : 6406.182646
selected by candidate: {'6405.61': 372, '6406.16': 28}
selected by status   : {'ok': 384, 'ok_static': 16}
error rows           : 0


In [5]:
try:
    import pandas as pd

    selected_df = pd.DataFrame(selected_rows)
    display(
        selected_df[
            ["task", "candidate", "status", "score", "cost", "memory", "params", "filesize", "file"]
        ].sort_values("task").reset_index(drop=True)
    )
except ImportError:
    selected_rows[:10]

,task,candidate,status,score,cost,memory,params,filesize,file
0,task001,6405.61,ok,17.102704,2690.0,2666,24,1172,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...
1,task002,6405.61,ok,13.632725,86446.0,86400,46,9145,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...
2,task003,6405.61,ok,17.918291,1190.0,1172,18,5000,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...
3,task004,6405.61,ok,14.354456,42005.0,40960,1045,3106,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...
4,task005,6406.16,ok,14.009146,59329.0,57419,1910,46553,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...
...,...,...,...,...,...,...,...,...,...
395,task396,6405.61,ok,14.617301,32296.0,31785,511,8843,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...
396,task397,6405.61,ok,15.105150,19828.0,19741,87,6674,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...
397,task398,6405.61,ok_static,14.569007,33894.0,33140,754,8089,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...
398,task399,6405.61,ok,17.923346,1184.0,912,272,1744,/Users/kaiikeda/Programming/Kaggle/Neuro Golf/...


In [6]:
write_submission_zip(selected_rows, OUTPUT_ZIP)
zip_report = validate_submission_zip(OUTPUT_ZIP)
zip_report

{'zip': '/Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/submission.zip',
 'exists': True,
 'size_bytes': 643670,
 'num_entries': 400,
 'num_task_files': 400,
 'has_directories': False,
 'has_duplicate_names': False,
 'invalid_names': [],
 'oversized_entries': []}

In [7]:
print(f"Final submission: {OUTPUT_ZIP}")
print(f"Final total score: {summary['total_score']:.6f}")

Final submission: /Users/kaiikeda/Programming/Kaggle/Neuro Golf/outputs/submission.zip
Final total score: 6406.182646


Notes:

- `outputs/submission.zip` is overwritten when the zip cell runs.
- `status == "ok_static"` means runtime profiling failed locally, so static shape memory was used as fallback.
- The zip contains only flat `taskXXX.onnx` entries, not candidate folder names.